In [3]:
import requests
import json
import pandas as pd
import time

In [ ]:
def OpenAlex_API():
    # each page is 25 records, so I will need to access 20,000 pages
    # I did a sample with 50 pages
    # https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/paging 
    for i in range(1, 50):
        filename = 'OpenAlex Data Batch 1' 
        url = 'https://api.openalex.org/works?page=' + str(i) + '&filter=type:types/dataset,authorships.institutions.lineage:i205783295|i2801744472|i170897317|i145311948|i2802508227|i57206974|i130769515|i20089843|i2801919071|i188538660|i79576946|i27837315|i2800403580|i4210130704|i114395901|i169521973|i135310074|i859038795|i204465549'
        r = requests.get(url)
        
        print(r.status_code)
        print(i)
        
        
    with open(filename, 'w', encoding='UTF-8') as fout:
        fout.write(r.text)
        print(i)
    
    time.sleep(3)
OpenAlex_API()


In [ ]:
def Read_JSON(input, output): 
    with open(input, encoding='UTF-8') as f:
        data = json.loads(f.read())
        df = pd.json_normalize(data, record_path=['results'])
    # https://docs.google.com/spreadsheets/d/1uRmsSanUZgdOJfMQzA1oYp1yx25NByg3qCdSdlMs1Yc/edit?usp=sharing 
    #file = df.to_csv(output)
df = Read_JSON('OpenAlex Data Batch 1', 'OpenAlex Data Batch 1.csv')
df = pd.read_csv('OpenAlex Data Batch 1.csv')
df


In [ ]:
import requests
import time
import json

def fetch_records(url, params=None, headers=None):
    """Fetch records from OpenAlex API."""
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an error for bad responses
    return response.json()

def OpenAlex_API2():
    url = 'https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i205783295|i2801744472|i170897317|i145311948|i2802508227|i57206974|i130769515|i20089843|i2801919071|i188538660|i79576946|i27837315|i2800403580|i4210130704|i114395901|i169521973|i135310074|i859038795|i204465549'
    headers = {
                'Content-Type': 'application/json'    
                }
    params = {
        "page_size": 10,
        }
    
    while True:
        filename = 'Open Alex Data Batch 2'
        data = fetch_records(url, params)
        works = data.get("results", [])
        
        for work in works:
            print(f"Title: {work.get('title', 'No Title')}")
            print(f"DOI: {work.get('doi', 'No DOI')}")
            print(f"Published Date: {work.get('publication_date', 'No Date')}")
            print("-" * 40) 
        print(len(work))
 
    
        next_cursor = data.get("meta", {}).get("next_cursor")
        print(next_cursor)
        with open(filename, 'w', encoding='UTF-8') as fout:
            fout.write(json.dumps(data))
        if not next_cursor:
            break
         
        params["cursor"] = next_cursor
        time.sleep(3)

OpenAlex_API2()    

In [ ]:
df = Read_JSON('Open Alex Data Batch 2', 'OpenAlex Data Batch 2.csv')
df = pd.read_csv('OpenAlex Data Batch 2.csv')
df

In [ ]:
import requests
import time
import json
# First attempt using ChatGPT
def fetch_openalex_data(url):
    page = 1
    while True:
        filename = 'Open Alex Data Batch 3'

        # Construct the full URL with the current page number
        current_url = f"{url}&page={page}&per-page=200"
        print(f"Fetching data from: {current_url}")
        
        # Send the GET request to the API
        response = requests.get(current_url)
        
        # Check for request success
        if response.status_code == 200:
            print(f"Status code: {response.status_code}", ';',  "Page number: ", page)
        if response.status_code != 200:
            print(f"Failed to fetch data: {response.status_code}")
            break
        
        # Parse the JSON response
        data = response.json()
        with open(filename, 'w', encoding='UTF-8') as fout:
                fout.write(json.dumps(data))    
        # Check if there are any works in the response
        if 'results' not in data or len(data['results']) == 0:
            print("No more results to fetch.")
            break
        record_number = 0
        for work in data['results']:
            #with open(filename, 'w', encoding='UTF-8') as fout:
                #fout.write(json.dumps(work))
            record_number += 1
            print("Page number: ", page, record_number, work)
            
    
        
        # Move to the next page
        page += 1
        
        
        time.sleep(3)


if __name__ == "__main__":
    #base_url = "https://api.openalex.org/works?filter=type:types/dataset,authorships.institutions.lineage:i205783295|i2801744472|i170897317|i145311948|i2802508227|i57206974|i130769515|i20089843|i2801919071|i188538660|i79576946|i27837315|i2800403580|i4210130704|i114395901|i169521973|i135310074|i859038795|i204465549"
    base_url = "https://api.openalex.org/works?filter=type:types/dataset,authorships.institutions.lineage:i205783295"
    fetch_openalex_data(base_url)


In [ ]:
df = Read_JSON('Open Alex Data Batch 3', 'Open Alex Data Batch 3.csv')
df = pd.read_csv('OpenAlex Data Batch 3.csv')

df
# loses results heirarchy 

In [ ]:
import time
import json
import requests
def OpenAlexAPI(url, params):
    
    # Initialize cursor
    cursor = "*"

    # Loop through pages
    all_datasets = []
    page = 0
    while cursor:
        params["cursor"] = cursor
        
        # Check for request success
        response = requests.get(url, params=params)
        if response.status_code == 200:
            print(f"Status code: {response.status_code}", ';',  "Page number: ", page)
        if response.status_code != 200:
            print("Oh no! Something went wrong during the live demo! How embarrassing!")
            response.raise_for_status()
        records = response.json()['results']
        record_number = 0
        page += 1
        for record in records:
            record_number += 1
            print("Page number: ", page, 'Record number:', record_number, record)
            all_datasets.append(record) 
            

        # Update cursor
        cursor = response.json()['meta']['next_cursor']
        
        time.sleep(3)
    return all_datasets
# types/dataset,
parameters = {
    #"filter": f"authorships.institutions.lineage:i205783295|i2801744472|i170897317|i145311948|i2802508227|i57206974|i130769515|i20089843|i2801919071|i188538660|i79576946|i27837315|i2800403580|i4210130704|i114395901|i169521973|i135310074|i859038795|i204465549",  # University of Tasmania
    "per-page": 100,
    #"select": "id,doi,publication_year,title,primary_location,authorships,topics",
}

OpenAlexDataset = OpenAlexAPI('https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i205783295', parameters)
#allrows = api_query_page_results('https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i205783295|i2801744472|i170897317|i145311948|i2802508227|i57206974|i130769515|i20089843|i2801919071|i188538660|i79576946|i27837315|i2800403580|i4210130704|i114395901|i169521973|i135310074|i859038795|i204465549', parameters)

In [ ]:
df = pd.DataFrame(OpenAlexDataset)
df.to_csv('OpenAlexData.csv')
df = pd.read_csv('OpenAlexData.csv')
df